**Finite Capacity Planning (FCP)**

This notebook explores the application of Finite Capacity Planning (FCP) using Python. This
a practical approach to optimize production scheduling by considering real-world
constraints such as machine availability, operation time, and order priorities.
Finite Capacity Planning is a method used in manufacturing and supply chain
management to create feasible production schedules by accounting for limited resources
(machines, labor, tools) and real-time capacity constraints.
The objectives of the methods presented in this Netbook are to:
1. Check whether available raw material stock is sufficient
2. Prioritize orders based on specific rules
3. Simulate various planning scenarios
4. Balance workload across machines
5. Create a planning performance dashboard

In [ ]:
import datetime
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from ortools.sat.python import cp_model
import seaborn as sns
from tqdm import tqdm

In [ ]:
# Loading data from the Excel file   file_path = "FCP_Planning.xlsx"
xls = pd.ExcelFile(file_path)
components_df = xls.parse("Components Length")
assemblies_df = xls.parse("Assemblies")
Products_df = xls.parse("Products")
Workstations_df = xls.parse("Workstations")
orders_df = xls.parse("Orders")
bom_df = xls.parse("Bill of Materials")
inventory_df = xls.parse("Inventory")
purchasing_df = xls.parse("Purchasing")

In [ ]:
# 1. Calculation of total requested length by product
def get_required_product_quantities(orders_df):
  """Return total quantity to produce per product, considering only undelivered orders."""
  required_products = (
      orders_df.groupby("Product ID")["Remaining Quantity"].sum().reset_index()
  )
  required_products = required_products[
      required_products["Remaining Quantity"] > 0
  ]
  return required_products

In [ ]:
required_products = get_required_product_quantities(orders_df)

In [ ]:
# 2. Bill of Materials (BOM) expansion
def get_required_items(bom_df, required_products):
  """Return required items and quantities by expanding BOM for required products."""
  required_items = bom_df.merge(
      required_products, on="Product ID", how="inner"
  )
  required_items["Required Quantity"] = (
      required_items["Quantity"] * required_items["Remaining Quantity"]
  )
  required_items = (
      required_items.groupby("Item ID")["Required Quantity"].sum().reset_index()
  )
  return required_items

In [ ]:
required_items = get_required_items(bom_df, required_items)

In [ ]:
# 3. Sub-assembly resolution
def resolve_assemblies(assemblies_df, required_items):
  assemblies = required_items.merge(
      assemblies_df, left_on="Item ID", right_on="Assembly ID", how="left"
  )
  non_assemblies = assemblies[assemblies["Component ID"].isna()][
      ["Item ID", "Required Quantity"]
  ]
  non_assemblies = non_assemblies.rename(columns={"Item ID": "Component ID"})

  assemblies = assemblies[assemblies["Component ID"].notna()]
  assemblies["Component Required Quantity"] = (
      assemblies["Required Quantity"] * assemblies["Quantity"]
  )
  assemblies_grouped = (
      assemblies.groupby("Component ID")["Component Required Quantity"]
      .sum()
      .reset_index()
  )
  assemblies_grouped = assemblies_grouped.rename(
      columns={"Component Required Quantity": "Required Quantity"}
  )

  required_components = pd.concat(
      [non_assemblies, assemblies_grouped], axis=0
  )
  required_components = (
      required_components.groupby("Component ID")["Required Quantity"]
      .sum()
      .reset_index()
  )
  return required_components

In [ ]:
required_components = resolve_assemblies(assemblies_df, required_items)

In [ ]:
# 4. Merge with component lengths and types
def get_required_components_with_length(components_df, required_components):
  """Merge required components with their type and length."""
  merged = required_components.merge(
      components_df, left_on="Component ID", right_on="Component ID", how="inner"
  )
  return merged

In [ ]:
components_with_length = get_required_components_with_length(
    components_df, required_components
)

In [ ]:
def group_by_component_type_and_sum_length(components_with_length):
  """Group total required lengths by component type."""
  components_with_length["Total Length"] = (
      components_with_length["Required Quantity"]
      * components_with_length["Length"]
  )
  grouped = (
      components_with_length.groupby("TYPE")["Total Length"]
      .sum()
      .reset_index()
  )
  return grouped

In [ ]:
total_mat = group_by_component_type_and_sum_length(components_with_length)

In [ ]:
# 5. Cleaning for cutting optimization
components_df = components_with_length[
    ["TYPE", "Length", "Required Quantity"]
].copy()
components_df["Length"] = components_df["Length"].round().astype(int)
components_df["Required Quantity"] = (
    components_df["Required Quantity"].astype(int)
)

In [ ]:
# 6. Cutting optimization using OR-Tools (CP-SAT)
def optimize_cutting_with_ortools(
    group, purchasing_df, verbose=False, show_cutting_plan=True
):
  component_type = group["TYPE"].iloc[0]
  standard_row = purchasing_df[purchasing_df["TYPE"] == component_type]
  if standard_row.empty:
    return pd.Series({
        "TYPE": component_type,
        "Bars Needed": "Missing Std Length",
        "Utilization (%)": 0,
        "Status": "No Std Length",
        "Parts Count": group["Required Quantity"].sum(),
        "Time (s)": 0,
        "Cutting Plan": "N/A",
        "Variables": 0,
        "Constraints": 0,
    })

  bar_length = int(standard_row["Standard Length"].values[0])
  kerf = 5  # Marge de découpe (chute de scie)

  required_lengths = []
  for _, row in group.iterrows():
    required_lengths += [int(round(row["Length"]))] * int(
        row["Required Quantity"]
    )

  num_parts = len(required_lengths)
  num_bars = num_parts  # Majoration de sécurité pour le nombre de barres disponibles

  model = cp_model.CpModel()
  x = {}
  for i in range(num_parts):
    for j in range(num_bars):
      x[i, j] = model.NewBoolVar(f"x_{i}_{j}")

  y = [model.NewBoolVar(f"y_{j}") for j in range(num_bars)]

  for i in range(num_parts):
    model.Add(sum(x[i, j] for j in range(num_bars)) == 1)

  for j in range(num_bars):
    model.Add(
        sum(
            x[i, j] * (required_lengths[i] + kerf) for i in range(num_parts)
        )
        <= bar_length * y[j]
    )

  for i in range(num_parts):
    for j in range(num_bars):
      model.Add(x[i, j] <= y[j])

  model.Minimize(sum(y))

  solver = cp_model.CpSolver()
  solver.parameters.max_time_in_seconds = 10
  solver.parameters.relative_gap_limit = 0.05
  status = solver.Solve(model)

  bars_used = 0
  total_used_length = 0
  cutting_plan = []

  for j in range(num_bars):
    if solver.Value(y[j]) == 1:
      cut_lengths = []
      bar_usage = 0
      for i in range(num_parts):
        if solver.Value(x[i, j]) == 1:
          l = required_lengths[i]
          cut_lengths.append(l)
          bar_usage += l + kerf
      total_used_length += bar_usage
      cutting_plan.append(cut_lengths)
      bars_used += 1

  utilization = (
      round(100 * total_used_length / (bars_used * bar_length), 1)
      if bars_used > 0
      else 0
  )

  plan_text = ""
  if show_cutting_plan and cutting_plan:
    for idx, cuts in enumerate(cutting_plan, 1):
      plan_text += f"Bar {idx}: " + ", ".join([str(l) for l in cuts]) + "\n"
  else:
    plan_text = "No solution"

  num_vars = num_parts * num_bars + num_bars
  num_constraints = num_parts + num_bars + num_parts * num_bars

  return pd.Series({
      "TYPE": component_type,
      "Bars Needed": (
          bars_used
          if status in (cp_model.OPTIMAL, cp_model.FEASIBLE)
          else "Failed"
      ),
      "Utilization (%)": utilization,
      "Status": (
          "Solved"
          if status == cp_model.OPTIMAL
          else ("Feasible" if status == cp_model.FEASIBLE else "Unsolved")
      ),
      "Parts Count": len(required_lengths),
      "Time (s)": round(solver.WallTime(), 2),
      "Cutting Plan": plan_text.strip(),
      "Variables": num_vars,
      "Constraints": num_constraints,
  })

In [ ]:
grouped = components_df.groupby("TYPE")
results = []
for type_name in tqdm(grouped.groups.keys(), desc="Optimizing", unit="type"):
  group = grouped.get_group(type_name)
  result = optimize_cutting_with_ortools(group, purchasing_df, verbose=True)
  results.append(result)

results_df = pd.DataFrame(results)
results_df.to_excel("Optimization_Report_with_Plan.xlsx", index=False)

In [ ]:
# 7. Comparison of bar requirements with inventory   
def compare_bar_needs_with_inventory(results_df, purchasing_df, inventory_df):
  inventory_df["Length in Stock"] = (
      inventory_df["Length in Stock"].round().astype(int)
  )
  inventory_df["Stock Quantity"] = (
      inventory_df["Stock Quantity"].fillna(0).astype(int)
  )

  results_df = results_df[results_df["Bars Needed"] != "Failed"].copy()
  results_df["Bars Needed"] = results_df["Bars Needed"].astype(int)

  merged = pd.merge(results_df, purchasing_df, on="TYPE", how="left")
  merged["Standard Length"] = merged["Standard Length"].astype(int)

  merged = pd.merge(
      merged,
      inventory_df,
      left_on=["TYPE", "Standard Length"],
      right_on=["TYPE", "Length in Stock"],
      how="left",
  )

  merged["Stock Quantity"] = merged["Stock Quantity"].fillna(0).astype(int)
  merged["Bars to Purchase"] = (
      merged["Bars Needed"] - merged["Stock Quantity"]
  ).clip(lower=0)

  result = merged[
      [
          "TYPE",
          "Standard Length",
          "Bars Needed",
          "Stock Quantity",
          "Bars to Purchase",
      ]
  ]
  return result.sort_values(by="Bars to Purchase", ascending=False)

In [ ]:
bar_stock_check_df = compare_bar_needs_with_inventory(
    results_df, purchasing_df, inventory_df
)
bar_stock_check_df.to_excel("Bar_Stock_Comparison.xlsx", index=False)
print(bar_stock_check_df)

In [ ]:
# 8. Verification of customer order feasibility based on inventory
def get_quantity_fulfillable_per_order(orders_df, bom_df, inventory_df):
  inventory_df["Length in Stock"] = (
      inventory_df["Length in Stock"].round().astype(int)
  )
  inventory_df["Stock Quantity"] = (
      inventory_df["Stock Quantity"].fillna(0).astype(int)
  )
  inventory = (
      inventory_df.groupby(["TYPE"])["Stock Quantity"].sum().reset_index()
  )

  undelivered_orders = orders_df[orders_df["Remaining Quantity"] > 0].copy()
  undelivered_orders = undelivered_orders[
      ["Product ID", "Remaining Quantity"]
  ]

  results = []
  for _, order in undelivered_orders.iterrows():
    product_id = order["Product ID"]
    qty_needed = int(order["Remaining Quantity"])

    bom = bom_df[bom_df["Product ID"] == product_id].copy()
    if bom.empty:
      results.append({
          "Product ID": product_id,
          "Quantity Needed": qty_needed,
          "Quantity Can Be Fulfilled": 0,
      })
      continue

    required_components = (
        bom.groupby(["Item ID"])["Quantity"]
        .sum()
        .reset_index()
        .rename(columns={"Item ID": "TYPE", "Quantity": "Quantity per Unit"})
    )

    merged = pd.merge(required_components, inventory, on="TYPE", how="left")
    merged["Stock Quantity"] = merged["Stock Quantity"].fillna(0).astype(int)
    merged["Units Possible"] = (
        merged["Stock Quantity"] // merged["Quantity per Unit"]
    )

    if merged.empty or merged["Units Possible"].min() == 0:
      qty_can_fulfill = 0
    else:
      qty_can_fulfill = min(qty_needed, merged["Units Possible"].min())

    results.append({
        "Product ID": product_id,
        "Quantity Needed": qty_needed,
        "Quantity Can Be Fulfilled": qty_can_fulfill,
    })

  return pd.DataFrame(results)

In [ ]:
result_df = get_quantity_fulfillable_per_order(orders_df, bom_df, inventory_df)
print(result_df)
result_df.to_excel("Fulfillable_Orders_Report.xlsx", index=False)